# Module 04 — Lazy Evaluation, Actions & Reuse

In [3]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs
#from shared.demo_data import user_post_activity

spark = begin_module('module-04-lazy-execution')
ui_links()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 10:21:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


{'master_ui': 'http://localhost:8080',
 'app_ui': 'http://localhost:4040',
 'worker_1_ui': 'http://localhost:8081',
 'worker_2_ui': 'http://localhost:8082',
 'app_ui_internal': 'http://jupyter-lab:4040'}

In [4]:
from pyspark.sql import functions as F, Window
trips = spark.read.parquet(*config.yellow_taxi_paths())
expensive = (
    trips.filter('total_amount > 10')
    .withColumn('rn', F.row_number().over(Window.partitionBy('VendorID').orderBy(F.desc('total_amount'))))
    .filter('rn <= 100')
    .groupBy('PULocationID')
    .agg(F.avg('total_amount').alias('avg_fare'), F.count('*').alias('n'))
)
expensive.explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (14)
+- HashAggregate (13)
   +- Exchange (12)
      +- HashAggregate (11)
         +- Project (10)
            +- Filter (9)
               +- Window (8)
                  +- WindowGroupLimit (7)
                     +- Sort (6)
                        +- Exchange (5)
                           +- WindowGroupLimit (4)
                              +- Sort (3)
                                 +- Filter (2)
                                    +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [VendorID#0, PULocationID#7, total_amount#16]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [IsNotNull(total_amount), GreaterThan(total_amount,10.0)]
ReadSchema: struct<VendorID:int,PULocationID:int,total_amount:double>

(2) Filter
Input [3]: [VendorID#0, PULocationID#7, total_amount#16]
Condition : (isnotnull(total_amount#16) AND (total_amount#16 > 10.0))

(3) Sort
Input [

## Each action can trigger a new job

In [5]:
_, t1 = timed_action(expensive)
_, t2 = timed_action(expensive, action='collect')
print(f'count={t1:.2f}s, collect={t2:.2f}s — check UI job list')

count=2.80s, collect=0.90s — check UI job list


## Branching DAG — duplicate work

In [6]:
branch_a = expensive.filter('n > 50')
branch_b = expensive.filter('avg_fare > 30')
_, ta = timed_action(branch_a)
_, tb = timed_action(branch_b)
print(f'without cache: a={ta:.2f}s b={tb:.2f}s (shared ancestor recomputed)')

without cache: a=0.84s b=0.84s (shared ancestor recomputed)


## Spark Observation (metrics without double full scan for logging)

In [7]:
from pyspark.sql import Observation
from pyspark.sql import functions as F
from shared.utils import drop_iceberg_table

TMP = "local.default.module04_observe_write"
drop_iceberg_table(spark, TMP)

obs = Observation("write_row_count")
observed = (
    trips.filter("total_amount > 50")
    .observe(obs, F.count(F.lit(1)).alias("rows"))
)

(
    observed.writeTo(TMP)
    .using("iceberg")
    .create()
)

print("observed rows (through filter):", obs.get["rows"])

[Stage 37:============================>                            (6 + 6) / 12]

observed rows (through filter): 622077


In [8]:
print("table count (sanity check):", spark.table(TMP).count())

table count (sanity check): 622077


## cache / persist

In [9]:
expensive.cache()
_, c1 = timed_action(expensive)
_, c2 = timed_action(expensive.filter('n > 10'))
expensive.unpersist()
print(f'with cache: first={c1:.2f}s second={c2:.2f}s')

with cache: first=1.65s second=0.35s


## checkpoint — cuts lineage (serialization cost)

In [10]:
ckpt_dir = str(config.CHECKPOINT_DIR)
spark.sparkContext.setCheckpointDir(ckpt_dir)
ck = expensive.checkpoint(eager=True)
ck.explain('formatted')
_, tc = timed_action(ck)
print(f'checkpointed action: {tc:.2f}s')

26/11/24 10:26:53 WARN SparkContext: Spark is not running in local mode, therefore the checkpoint directory must not be on the local filesystem. Directory '/workspace/data/output/checkpoints' appears to be on the local filesystem.


== Physical Plan ==
* Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [3]: [PULocationID#7, avg_fare#22, n#23L]
Arguments: [PULocationID#7, avg_fare#22, n#23L], MapPartitionsRDD[108] at checkpoint at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)


checkpointed action: 0.10s


## Intermediate Iceberg table vs cache

In [12]:
drop_iceberg_table(spark, 'local.default.lazy_intermediate')
expensive.writeTo('local.default.lazy_intermediate').using('iceberg').create()
from_table = spark.table('local.default.lazy_intermediate')
from_table.explain('formatted')  # file scan + stats, fresh optimizer view
_, tt = timed_action(from_table)
print(f'read table: {tt:.2f}s')

== Physical Plan ==
* ColumnarToRow (2)
+- BatchScan local.default.lazy_intermediate (1)


(1) BatchScan local.default.lazy_intermediate
Output [3]: [PULocationID#444, avg_fare#445, n#446L]
IcebergScan(table=local.default.lazy_intermediate, schemaId=0, snapshotId=3892988405223367508, branch=null, filters=, runtimeFilters=, groupedBy=)

(2) ColumnarToRow [codegen id : 1]
Input [3]: [PULocationID#444, avg_fare#445, n#446L]


read table: 0.05s
